# Pruning - remove unused signals

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khanhnd61-vr/modelopt-101/blob/main/examples/03_pruning.ipynb)

**Goal.** Remove low-importance weights and filters from a trained model, then measure what
is actually gained. The lab compares two pruning granularities:

- **Unstructured pruning** zeros individual weights. It creates sparsity, but a dense tensor
  keeps its original shape, file size, and dense-kernel latency.
- **Structured pruning** removes complete filters and rebuilds narrower layers. It changes
  tensor shapes, so parameter count, file size, and latency can genuinely fall.

The compact experiment includes:

1. an unstructured accuracy sweep at `[0, 50, 70, 80, 90, 95]%` sparsity;
2. a structured raw-pruning sweep at `[30, 50, 70]%` filter removal;
3. one four-epoch fine-tune of the `50%` structured candidate;
4. a final comparison of accuracy, parameters, size, and CPU latency.

**Runtime:** set `Runtime -> Change runtime type -> GPU`, then `Runtime -> Run all`.

## 1. Setup

On Colab `torch`/`torchvision` are pre-installed, so nothing to `pip install`.

In [ ]:
import io, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T

GLOBAL_SEED = 0
torch.manual_seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# --- training and pruning budgets ---
EPOCHS = 12
FINETUNE_EPOCHS = 4
BATCH_SIZE = 128
LR = 0.05

PRUNE_RATIO = 0.5
UNSTRUCTURED_RATIOS = [0.0, 0.5, 0.7, 0.8, 0.9, 0.95]
STRUCTURED_RATIOS = [0.3, 0.5, 0.7]

## 2. Data - MNIST

10 classes (handwritten digits 0-9) of 28x28 grayscale images. Small and quick to download.

In [ ]:
mean = (0.1307,)
std  = (0.3081,)

train_tf = T.Compose([
    T.RandomCrop(28, padding=2),
    T.ToTensor(),
    T.Normalize(mean, std),
])
test_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])

train_set = torchvision.datasets.MNIST("./data", train=True,  download=True, transform=train_tf)
test_set  = torchvision.datasets.MNIST("./data", train=False, download=True, transform=test_tf)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=256,        shuffle=False, num_workers=2)

classes = train_set.classes
print(len(train_set), "train /", len(test_set), "test images")

## 3. The model

A VGG-style CNN whose channel widths are a parameter. That matters for pruning: *structured*
pruning will rebuild the **same network with narrower layers**, and `widths` is exactly what
shrinks.

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
    )

class Net(nn.Module):               # default widths -> ~1.1M params
    def __init__(self, widths=(64, 64, 128, 128, 256, 256), num_classes=10):
        super().__init__()
        w = widths
        self.features = nn.Sequential(
            conv_block(1,    w[0]), conv_block(w[0], w[1]), nn.MaxPool2d(2),   # 28 -> 14
            conv_block(w[1], w[2]), conv_block(w[2], w[3]), nn.MaxPool2d(2),   # 14 -> 7
            conv_block(w[3], w[4]), conv_block(w[4], w[5]), nn.MaxPool2d(2),   # 7  -> 3
        )
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(w[5], num_classes))
        self.widths = tuple(widths)

    def forward(self, x):
        return self.head(self.features(x))

def count_params(m):
    return sum(p.numel() for p in m.parameters())

# the six conv blocks live at these indices inside `features` (the rest are MaxPool)
CONV_IDX = [0, 1, 3, 4, 6, 7]
def conv_blocks(model):
    return [model.features[i] for i in CONV_IDX]

print(f"params: {count_params(Net()):,}")

## 4. Training, evaluation, and metric utilities

The dense model is trained once. Every pruning result starts from that same checkpoint.
Latency is measured with batch size one on CPU, while model size is the serialized PyTorch
state dictionary.

In [ ]:
def train(model, epochs=EPOCHS, lr=LR, tag=""):
    model.to(device).train()
    optimizer = torch.optim.SGD(
        model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    for epoch in range(epochs):
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = F.cross_entropy(model(x), y)
            loss.backward()
            optimizer.step()
        scheduler.step()
        print(
            f"[{tag}] epoch {epoch + 1:02d}/{epochs} "
            f"test_acc={evaluate(model):.2f}%"
        )
    return model


@torch.no_grad()
def evaluate(model):
    was_training = model.training
    model.eval()
    correct = total = 0
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total += y.numel()
    if was_training:
        model.train()
    return 100.0 * correct / total


@torch.no_grad()
def latency_ms(model, n=100):
    """Average single-image CPU inference latency in milliseconds."""
    model_cpu = copy.deepcopy(model).to("cpu").eval()
    x = torch.randn(1, 1, 28, 28)
    for _ in range(10):
        model_cpu(x)
    start = time.perf_counter()
    for _ in range(n):
        model_cpu(x)
    return (time.perf_counter() - start) / n * 1000.0


def model_size_mb(model):
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return buf.getbuffer().nbytes / 1e6

## 5. Train the dense baseline

Train the full-width network normally. This is the **"before"** model we prune.

In [ ]:
model_dense = train(Net(), tag="dense")
dense_acc = evaluate(model_dense)
print(f"\nDense baseline accuracy: {dense_acc:.2f}%")

## 6. Unstructured pruning - global magnitude criterion

All convolution weights are ranked together by absolute value. The threshold is the
requested quantile, and values below it are set to zero.

This is **global** pruning: sensitive and insensitive layers do not have to lose the same
percentage. Only convolution weights are pruned; BatchNorm and classifier values stay dense.

In [ ]:
def prune_unstructured(model, ratio):
    """Zero globally smallest-magnitude conv weights in a copied model."""
    if not 0.0 <= ratio < 1.0:
        raise ValueError("ratio must be in [0, 1)")

    masked_model = copy.deepcopy(model)
    all_weights = torch.cat([
        block[0].weight.data.abs().flatten()
        for block in conv_blocks(masked_model)
    ])
    threshold = torch.quantile(all_weights, ratio)

    with torch.no_grad():
        for block in conv_blocks(masked_model):
            weight = block[0].weight.data
            weight.mul_((weight.abs() > threshold).to(weight.dtype))

    return masked_model, threshold.item()


masked, threshold_50 = prune_unstructured(model_dense, PRUNE_RATIO)
nonzero_50 = sum(
    (block[0].weight.data != 0).sum().item()
    for block in conv_blocks(masked)
)
total_conv_weights = sum(
    block[0].weight.numel() for block in conv_blocks(masked)
)
masked_sparsity = 1.0 - nonzero_50 / total_conv_weights

print(f"criterion=|w| ratio={PRUNE_RATIO:.0%} threshold={threshold_50:.6f}")
print(
    f"actual conv sparsity: {masked_sparsity:.2%} "
    f"({total_conv_weights - nonzero_50:,}/{total_conv_weights:,} weights zero)"
)

import matplotlib.pyplot as plt

all_dense_weights = torch.cat([
    block[0].weight.data.abs().flatten()
    for block in conv_blocks(model_dense)
]).cpu()

plt.figure(figsize=(8, 3.5))
plt.hist(all_dense_weights.numpy(), bins=100, color="#777777")
plt.axvline(
    threshold_50,
    color="#d1495b",
    linewidth=2,
    label=f"prune below {threshold_50:.4f}",
)
plt.xlabel("absolute weight value")
plt.ylabel("count")
plt.title("Global magnitude threshold at 50% pruning")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Unstructured sweep - accuracy versus sparsity

The existing sweep is inexpensive because it requires no retraining. It reveals where raw
magnitude pruning stops being forgiving. The tensors remain dense at every ratio, even when
most values are zero.

In [ ]:
unstructured_results = []
for ratio in UNSTRUCTURED_RATIOS:
    model_masked, threshold = prune_unstructured(model_dense, ratio)
    nonzero = sum(
        (block[0].weight.data != 0).sum().item()
        for block in conv_blocks(model_masked)
    )
    total = sum(block[0].weight.numel() for block in conv_blocks(model_masked))
    unstructured_results.append({
        "ratio": ratio,
        "actual_sparsity": 1.0 - nonzero / total,
        "threshold": threshold,
        "accuracy": evaluate(model_masked.to(device)),
    })

print(f"{'target':>10}{'actual':>11}{'threshold':>13}{'accuracy':>12}")
print("-" * 46)
for result in unstructured_results:
    print(
        f'{result["ratio"]:>9.0%}{result["actual_sparsity"]:>11.2%}'
        f'{result["threshold"]:>13.6f}{result["accuracy"]:>11.2f}%'
    )

plt.figure(figsize=(8, 4))
plt.plot(
    [result["actual_sparsity"] * 100 for result in unstructured_results],
    [result["accuracy"] for result in unstructured_results],
    "o-",
    color="#d1495b",
    linewidth=2,
)
plt.xlabel("actual convolution-weight sparsity (%)")
plt.ylabel("test accuracy (%)")
plt.title("Unstructured magnitude pruning without fine-tuning")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nZeros remain inside dense tensors: dense file size and kernels do not shrink.")

## 8. Structured filter pruning - 30%, 50%, and 70%

Each convolution filter is scored by its L1 norm. The top `1-ratio` filters are retained,
and a narrower network is rebuilt with consistent input/output channel wiring.

All three ratios are first measured **without fine-tuning**. This isolates the immediate
effect of removing filters and makes the ratio trade-off visible at low runtime cost.

In [ ]:
def structured_prune(model, ratio):
    """Remove low-L1 filters and return a physically narrower copied network."""
    if not 0.0 <= ratio < 1.0:
        raise ValueError("ratio must be in [0, 1)")

    source = copy.deepcopy(model).to("cpu").eval()
    source_blocks = conv_blocks(source)
    kept_indices = []

    for block in source_blocks:
        n_filters = block[0].weight.shape[0]
        n_keep = max(1, round(n_filters * (1.0 - ratio)))
        l1_scores = block[0].weight.data.abs().sum(dim=(1, 2, 3))
        indices = torch.topk(l1_scores, n_keep).indices.sort().values
        kept_indices.append(indices)

    pruned_model = Net(
        widths=tuple(len(indices) for indices in kept_indices)
    ).to("cpu").eval()
    previous_indices = torch.arange(1)

    with torch.no_grad():
        for source_block, target_block, indices in zip(
            source_blocks, conv_blocks(pruned_model), kept_indices
        ):
            target_block[0].weight.copy_(
                source_block[0].weight[indices][:, previous_indices, :, :]
            )
            target_block[1].weight.copy_(source_block[1].weight[indices])
            target_block[1].bias.copy_(source_block[1].bias[indices])
            target_block[1].running_mean.copy_(source_block[1].running_mean[indices])
            target_block[1].running_var.copy_(source_block[1].running_var[indices])
            target_block[1].num_batches_tracked.copy_(
                source_block[1].num_batches_tracked
            )
            previous_indices = indices

        pruned_model.head[2].weight.copy_(
            source.head[2].weight[:, previous_indices]
        )
        pruned_model.head[2].bias.copy_(source.head[2].bias)

    return pruned_model


structured_models = {}
structured_raw_results = []

print(
    f"{'ratio':>8}{'widths':>31}{'params':>13}"
    f"{'shape reduction':>18}{'accuracy':>12}"
)
print("-" * 82)
for ratio in STRUCTURED_RATIOS:
    model_structured = structured_prune(model_dense, ratio)
    structured_models[ratio] = model_structured
    params = count_params(model_structured)
    accuracy = evaluate(model_structured.to(device))
    result = {
        "ratio": ratio,
        "model": model_structured,
        "params": params,
        "shape_reduction": 1.0 - params / count_params(model_dense),
        "accuracy": accuracy,
    }
    structured_raw_results.append(result)
    print(
        f'{ratio:>7.0%}{str(model_structured.widths):>31}{params:>13,}'
        f'{result["shape_reduction"]:>17.2%}{accuracy:>11.2f}%'
    )

## 9. Fine-tune the 50% structured candidate

Only the middle candidate is fine-tuned. It represents a useful balance: enough pruning to
produce a clear size/latency reduction without choosing the most aggressive 70% setting.

The raw 50% measurements are retained, so the accuracy recovered by four fine-tuning epochs
can be reported directly.

In [ ]:
structured_50_raw = structured_models[PRUNE_RATIO]
structured_50_raw_acc = next(
    result["accuracy"]
    for result in structured_raw_results
    if result["ratio"] == PRUNE_RATIO
)

structured_50_finetuned = train(
    copy.deepcopy(structured_50_raw),
    epochs=FINETUNE_EPOCHS,
    lr=LR * 0.2,
    tag="structured-50-finetune",
)
structured_50_finetuned_acc = evaluate(structured_50_finetuned)

# Preserve the concise variable name used by the original lab.
pruned = structured_50_finetuned
pruned_acc = structured_50_finetuned_acc

print(f"\n50% structured accuracy before fine-tune: {structured_50_raw_acc:.2f}%")
print(f"50% structured accuracy after fine-tune:  {pruned_acc:.2f}%")
print(f"accuracy recovered: {pruned_acc - structured_50_raw_acc:+.2f}pp")

## 10. Final comparison - what actually becomes smaller?

The final table includes the dense model, the 50% unstructured model, all three raw
structured candidates, and the fine-tuned 50% candidate.

`shape reduction` measures deleted parameters. It stays at zero for unstructured pruning
because zero-valued weights still occupy positions in the original dense tensors.

In [ ]:
masked_acc = evaluate(masked.to(device))
dense_params = count_params(model_dense)

def collect_result(name, model, target_ratio, accuracy, stage):
    params = count_params(model)
    return {
        "name": name,
        "model": model,
        "target_ratio": target_ratio,
        "stage": stage,
        "params": params,
        "shape_reduction": 1.0 - params / dense_params,
        "size_mb": model_size_mb(model),
        "accuracy": accuracy,
        "delta_pp": accuracy - dense_acc,
        "latency_ms": latency_ms(model),
    }


final_results = [
    collect_result("Dense", model_dense, 0.0, dense_acc, "trained"),
    collect_result("Unstruct 50", masked, 0.5, masked_acc, "raw"),
]

for raw in structured_raw_results:
    ratio = raw["ratio"]
    final_results.append(collect_result(
        f"Struct {ratio:.0%} raw",
        raw["model"],
        ratio,
        raw["accuracy"],
        "raw",
    ))

final_results.append(collect_result(
    "Struct 50 FT",
    structured_50_finetuned,
    PRUNE_RATIO,
    structured_50_finetuned_acc,
    "fine-tuned",
))

print(
    f"{'model':<17}{'target':>9}{'stage':>12}{'params':>13}"
    f"{'shape red.':>13}{'size':>10}{'accuracy':>11}"
    f"{'vs dense':>11}{'latency':>12}"
)
print("-" * 108)
for result in final_results:
    print(
        f'{result["name"]:<17}{result["target_ratio"]:>9.0%}'
        f'{result["stage"]:>12}{result["params"]:>13,}'
        f'{result["shape_reduction"]:>12.1%}{result["size_mb"]:>8.2f}MB'
        f'{result["accuracy"]:>10.2f}%{result["delta_pp"]:>+10.2f}pp'
        f'{result["latency_ms"]:>10.2f}ms'
    )

names = [result["name"] for result in final_results]
colors = ["#4c4c4c", "#d1495b", "#6a994e", "#2a9d8f", "#457b9d", "#f4a261"]
x = np.arange(len(final_results))

metric_plots = [
    ("accuracy", "Test accuracy", "%"),
    ("params", "Parameter count", ""),
    ("size_mb", "Serialized model size", "MB"),
    ("latency_ms", "CPU latency / image", "ms"),
]

fig, axes = plt.subplots(2, 3, figsize=(19, 10))
for axis, (key, title, unit) in zip(axes.flat[:4], metric_plots):
    values = [result[key] for result in final_results]
    bars = axis.bar(x, values, color=colors)
    axis.set_xticks(x, names, rotation=18)
    axis.set_title(title)
    axis.grid(axis="y", alpha=0.2)
    if key == "accuracy":
        axis.set_ylim(max(0, min(values) - 3), min(100, max(values) + 1))
    for bar, value in zip(bars, values):
        label = f"{value / 1e6:.2f}M" if key == "params" else f"{value:.2f}{unit}"
        axis.text(
            bar.get_x() + bar.get_width() / 2,
            value,
            label,
            ha="center",
            va="bottom",
            fontsize=8,
        )

axis = axes[1, 1]
axis.plot(
    [result["actual_sparsity"] * 100 for result in unstructured_results],
    [result["accuracy"] for result in unstructured_results],
    "o-",
    color="#d1495b",
    linewidth=2,
)
axis.set_xlabel("unstructured sparsity (%)")
axis.set_ylabel("accuracy (%)")
axis.set_title("Unstructured accuracy cliff")
axis.grid(alpha=0.25)

axis = axes[1, 2]
for result, color in zip(final_results[2:], colors[2:]):
    axis.scatter(
        result["size_mb"],
        result["accuracy"],
        s=110,
        color=color,
    )
    axis.annotate(
        result["name"],
        (result["size_mb"], result["accuracy"]),
        xytext=(5, 6),
        textcoords="offset points",
        fontsize=9,
    )
axis.set_xlabel("model size (MB) - lower is better")
axis.set_ylabel("accuracy (%) - higher is better")
axis.set_title("Structured size/accuracy trade-off")
axis.grid(alpha=0.25)

plt.suptitle("Dense vs. unstructured vs. structured pruning", fontsize=15)
plt.tight_layout()
plt.show()

dense_result = final_results[0]
unstructured_result = final_results[1]
structured_ft_result = final_results[-1]
raw_70_result = next(
    result for result in final_results
    if result["name"] == "Struct 70% raw"
)

print("\nFINAL EVALUATION")
print("=" * 78)
print(
    f'Unstructured 50% creates {masked_sparsity:.1%} conv sparsity but '
    f'{unstructured_result["shape_reduction"]:.1%} shape reduction; its dense file size '
    f'is {unstructured_result["size_mb"] / dense_result["size_mb"]:.2f}x the baseline.'
)
print(
    f'Structured 50% + fine-tuning is '
    f'{dense_result["size_mb"] / structured_ft_result["size_mb"]:.2f}x smaller and '
    f'{dense_result["latency_ms"] / structured_ft_result["latency_ms"]:.2f}x faster, '
    f'with {structured_ft_result["delta_pp"]:+.2f}pp accuracy change.'
)
print(
    f'Fine-tuning recovered '
    f'{structured_50_finetuned_acc - structured_50_raw_acc:+.2f}pp for the 50% candidate.'
)
print(
    f'The aggressive 70% raw model achieves '
    f'{raw_70_result["shape_reduction"]:.1%} parameter-shape reduction but changes '
    f'accuracy by {raw_70_result["delta_pp"]:+.2f}pp before fine-tuning.'
)

## 11. Takeaways

- **Criterion:** global weight magnitude is a simple unstructured baseline; per-filter L1
  norm provides a natural structured score.
- **Granularity:** zeros do not make a dense tensor smaller. Removing complete channels does.
- **Ratio:** 30%, 50%, and 70% expose the compression/accuracy curve without a long sweep.
- **Recovery:** the raw and fine-tuned 50% rows isolate how much accuracy four additional
  epochs recover.
- **Latency:** structured pruning can accelerate ordinary dense kernels because layer shapes
  are smaller. Unstructured speedup requires sparse storage and sparse kernel support.

Useful extensions are random-filter controls, per-layer versus global thresholds, and
iterative prune/fine-tune cycles. For an edge pipeline, a common order is **distill, prune,
then quantize** the surviving compact model.